# Effective ownership
**Question:** how much does each owner really own of each company, once we follow every chain of companies in between?

- **Input:** current ownership links from `silver.ownership`.
- **Output:** `eval.eff_ownership`, one row per owner and company, with the effective share and a flag when the data behind it conflicts.
- **How to run:** Run all, top to bottom. Each step depends on the one before.

Example: A owns 50% of B, and B owns 40% of C. Through B, A effectively owns **50% × 40% = 20%** of C. If A also reaches C another way (100% of D, which owns 10% of C), the routes add up: 20% + 10% = **30%**.

### Step 1: ownership links with a percentage
Only links with a known percentage can be multiplied, so links without one are left out here. GLEIF links are also left out: GLEIF records which company tops a group in the accounts, not who holds the shares, and mixing it with real shareholdings counted some stakes twice. Result: 71,026 links.

In [0]:
CREATE OR REPLACE  TABLE pep_dev.eval.edges_pct
USING DELTA
AS
SELECT owner_id,asset_id ,owner_type,asset_type, MAX(pct_min)/100 AS share FROM pep_dev.silver.ownership
WHERE 
    pct_min IS NOT NULL 
    AND is_current 
    AND  NOT is_self_owning 
    AND NOT array_contains(datasets, 'ext_gleif')
GROUP BY owner_id, asset_id, owner_type, asset_type;

SELECT count(*) AS edges, max(share) AS max_share FROM pep_dev.eval.edges_pct

### Step 2: start the routes
Every direct holding is a route of one step. `path` keeps the list of companies walked so far, top first. This cell also resets the table, so the loop below always starts fresh.

In [0]:
CREATE OR REPLACE TABLE pep_dev.eval.paths
USING DELTA 
AS
SELECT owner_id as  top_id,owner_type as top_type, asset_id, share, 1 AS depth, array(owner_id, asset_id) AS path 
FROM pep_dev.eval.edges_pct;

### Step 3: walk down the chains
Each round takes every route one step further, multiplies the shares and adds the new company to the end of the path. A company already on a route is never visited again, so ownership loops can't go round forever. The walk stops by itself when a round finds nothing new: the longest chain is 18 links, and there are 256,902 routes in total.

**Running it twice without the reset duplicates every route.**

In [0]:
    %python

    for r in range(2,31):
        res=spark.sql(""" 
                INSERT INTO pep_dev.eval.paths
                SELECT 
                p.top_id , 
                p.top_type, 
                e.asset_id,
                p.share*e.share,
                
                :r,
                array_append(p.path,e.asset_id)
                
                

                FROM pep_dev.eval.paths as p
                JOIN pep_dev.eval.edges_pct as e
                ON p.asset_id = e.owner_id
            
                WHERE p.depth= :r-1 AND 
                NOT array_contains(p.path,e.asset_id)

                
                """, args={"r":r})
        n=res.collect()[0]["num_inserted_rows"]
        print(f"depth {r}: {n} new paths")
        if n == 0:
            break


In [0]:
SELECT count(*), min(depth), max(depth) FROM pep_dev.eval.paths

### Over-owned companies
Some companies have current owners that add up to **more than 100%**. That can't happen in reality, so the source records contradict each other.

**Example (a shortcut):** P owns 100% of M, and M owns 100% of C. One source also records P as owning 100% of C directly:

| Owner | Company | Share | Note |
|---|---|---|---|
| P | M | 100% | real |
| M | C | 100% | real |
| P | C | 100% | shortcut: P is the top boss, not a direct shareholder |

C's owners add up to **M 100% + P 100% = 200%**. When we walk the chains, P reaches C twice:
- directly: 100%
- through M: 100% × 100% = 100%

So P's effective share in C comes out as **200%**, the same stake counted twice. This table catches C, with `total_share` = 2.0 and `n_owners` = 2.

**The three causes we found:**
- **Duplicate IDs:** the same owner appears under two IDs, each holding 100%.
- **Unfinished handovers:** the company changed hands, but the old owner's link was never closed.
- **Shortcuts:** like the example above.

Instead of guessing which record is right, we **flag** every route through these companies in `eff_ownership` (`has_conflict`). We use 1.01 rather than 1.0, so rounding (like 33.4 + 33.3 + 33.4) isn't flagged.

In [0]:
-- Companies whose current owners add up to more than 100% (data conflicts); used to flag routes.
CREATE OR REPLACE TABLE pep_dev.eval.over_owned
USING DELTA AS 
SELECT 
    asset_id,
    SUM(share) AS total_share,
    count(owner_id) as n_owners
FROM pep_dev.eval.edges_pct
GROUP BY asset_id 
HAVING SUM(share)>1.01;

-- SELECT * FROM pep_dev.eval.over_owned;

### Step 4: add up the routes
All routes from the same owner to the same company are added into one effective share. `n_paths` says how many routes there were. `has_conflict` is true if any route passes through an over-owned company, which means the share can't be trusted as is.

In [0]:
CREATE OR REPLACE TABLE pep_dev.eval.eff_ownership
USING DELTA 
SELECT 
    top_id,
    top_type,
    asset_id,
    SUM(share) as eff_share,
    count(*) as n_paths,
    MIN(depth) as min_depth,
    MAX(depth) as max_depth,

    bool_or(
        arrays_overlap(
        slice(path,2,size(path)),
        (SELECT collect_list(asset_id) FROM pep_dev.eval.over_owned)
        )
    )
     as has_conflict      -- has_conflict: true if any route passes through a company whose owners add up to >100% (over_owned). Top skipped.

FROM pep_dev.eval.paths

GROUP BY 
    top_id,
    top_type,
    asset_id;




### Check: every impossible share is flagged
A share above 100% can only come from conflicting data, so every one should carry the flag. Result: 90 pairs above 100% (max 299%), 2,172 pairs flagged in total, and none above 101% left unflagged. Four pairs reach up to 100.7% from rounding in the source data.

In [0]:
SELECT 
    count_if(eff_share>1.0001) AS over_100,
    max(eff_share) AS max_share,
    count_if(has_conflict) as flagged,
    count_if(eff_share>1.0001 AND NOT has_conflict) AS unflagged_over_100

FROM pep_dev.eval.eff_ownership;



### Known limits
- About a third of current links have no percentage, so they're missing from these totals.
- Where one owner holds two separate blocks in the same company, we keep the larger block, so those shares come out slightly low.
- In ownership loops, we never revisit a company. The strictly correct answer would count each extra lap with ever-smaller shares, but loops are a tiny part of the data.
- Conflicts are flagged, not fixed. Duplicate IDs should mostly disappear after entity resolution (Phase 3).

In [0]:
SELECT 
e.min_depth,
count(*),
count_if(e.eff_share>=0.5),
count_if(has_conflict),
max(e.eff_share)
FROM pep_dev.eval.eff_ownership e
JOIN pep_dev.eval.seeds s
ON e.top_id = s.node_id
GROUP BY e.min_depth
ORDER BY e.min_depth DESC    